### Config

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from hydra import compose, initialize
from pathlib import Path
from omegaconf import OmegaConf

initialize(config_path="../config", version_base="1.3")
cfg = compose(config_name="config")
print(OmegaConf.to_yaml(cfg))

sys.path.insert(0, str(cfg.paths.project_root))

paths:
  project_root: /home/p84400019/projects/consciousness-llms/IT-LLMs/
  model_path: ${model.company}/${model.model_family}/${model.model_size}/${model.it}/
  data_dir: ${paths.project_root}data/${paths.model_path}
  data_activations_dir: ${paths.data_dir}activations/
  data_activations_file: ${paths.data_activations_dir}multi_prompt_activations.pkl
  data_phyid_dir: ${paths.data_dir}phyid/
  data_phyid_file: ${paths.data_phyid_dir}multi_prompt_phyid.pkl
  plot_dir: ${paths.project_root}plots/${paths.model_path}
  plot_activations_dir: ${paths.plot_dir}activations/
  plot_time_series_dir: ${paths.plot_activations_dir}time_series/
  plot_phyid_dir: ${paths.plot_dir}phyid/
model:
  shortcode: D2-16-A2
  hf_name: deepseek-ai/DeepSeek-V2-Lite
  company: deepseek
  model_family: deepseek-v2
  model_size: 16B-A2B
  it: base
  plot_name: DeepSeek V2 Lite
  color: '#8e9e00'
  apply_chat_template: 'no'
generation:
  max_new_tokens: 128
  default_prompts:
  - Imagine a future where humans h

### Time Series Loading

In [2]:
from src.activation_recorder import MultiPromptActivations

data_activations_file = cfg.paths.data_activations_file
activations = MultiPromptActivations.load(file_path=data_activations_file)

/home/p84400019/miniconda3/envs/int/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MultiPromptActivations successfully loaded from '/home/p84400019/projects/consciousness-llms/IT-LLMs/data/deepseek/deepseek-v2/16B-A2B/base/activations/multi_prompt_activations.pkl'.


In [ ]:
from src.time_series_activations import MultiPromptTimeSeries

time_series = MultiPromptTimeSeries.from_activations(
    activations, 
    node_type=cfg.time_series.node_type,
    node_activation=cfg.time_series.node_activation,
    projection_method=cfg.time_series.projection_method, 
    exclude_shared_expert_moe=cfg.time_series.exclude_shared_expert_moe, 
)
time_series.plot(token_x=True, ticks_all_layers=True, plot_dir=cfg.paths.plot_time_series_dir)

### PhyID Decomposition

In [ ]:
from src.phyid_decomposition import MultiPromptPhyID, PromptPhyID, PhyIDTimeSeries
compute_phyid = True
data_phyid_file = cfg.paths.data_phyid_file

if compute_phyid:
    phyid_comp = MultiPromptPhyID.from_time_series(time_series) 
    phyid_comp.save(file_path=data_phyid_file)

In [ ]:
phyid = MultiPromptPhyID.load(file_path=data_phyid_file)
phyid.compute_extra_atoms()
phyid = phyid.get_prompt(prompt_index=0)  # Get the first prompt's phyid
phyid.build_data_array()

In [10]:
plot_dir = cfg.paths.plot_phyid_dir
# plot_dir = None
phyid.node_heatmap(atom='sts', plot_dir=plot_dir)
phyid.plot_mean_along('sts', varying_dim='source_layer', plot_dir=plot_dir)
phyid.plot_mean_along('sts_normalized', varying_dim='source_layer', plot_dir=plot_dir)
phyid.plot_mean_along('mutual_information', varying_dim='source_layer', plot_dir=plot_dir)

Heatmap saved to /home/p84400019/projects/consciousness-llms/IT-LLMs/plots/deepseek/deepseek-v2/16B-A2B/base/phyid/node_heatmap/sts/heatmap.png
Plot saved to /home/p84400019/projects/consciousness-llms/IT-LLMs/plots/deepseek/deepseek-v2/16B-A2B/base/phyid/plot_mean_along/sts/source_layer.png
Plot saved to /home/p84400019/projects/consciousness-llms/IT-LLMs/plots/deepseek/deepseek-v2/16B-A2B/base/phyid/plot_mean_along/sts_normalized/source_layer.png
Plot saved to /home/p84400019/projects/consciousness-llms/IT-LLMs/plots/deepseek/deepseek-v2/16B-A2B/base/phyid/plot_mean_along/mutual_information/source_layer.png
